# xarray

> Utilities for handling Euclid FITS images as xarray Datasets, using dask for efficient computation and memory usage.

In [ ]:
# | default_exp euclid.xarray

In [ ]:
# | export

import json
from warnings import catch_warnings, filterwarnings

from astropy.io.fits import HDUList, ImageHDU, PrimaryHDU
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
import pandas as pd
import xarray as xr
from kerchunk.combine import MultiZarrToZarr
from kerchunk.fits import process_file
from tqdm import tqdm

from nicl.euclid.utilities import (
    get_dither_id_from_filename,
    get_filter_from_filename,
    get_obs_id_from_filename,
    get_tile_index_from_filename,
)

In [ ]:
# | hide
# # Additional imports used in the examples

from astropy.wcs import WCS

from nicl.euclid.utilities import default_data_path, get_nisp_images_for_observation

In [ ]:
# | exporti


def _fix_byte_order(refs):
    """Correctly identify the byte order as bigendian.

    This is a workaround. A proper fix has been submitted via this PR:
    https://github.com/fsspec/kerchunk/pull/531
    When that PR is merged this function will be unnecessary, but not cause any harm.
    """
    for k in refs.keys():
        if "zarray" in k:
            z = json.loads(refs[k])
            z["dtype"] = np.dtype(z["dtype"]).newbyteorder(">").str
            refs[k] = json.dumps(z).encode()
    return refs

In [ ]:
# | exporti


def _rename(refs, new_name):
    """Rename the zarray in a kerchunk json reference."""
    new_refs = {}
    for key in refs.keys():
        if "/" in key:
            parts = key[key.find("/") + 1 :]
            new_key = f"{new_name}/{parts}"
            new_refs[new_key] = refs[key]
    return new_refs

In [ ]:
# | exporti


def _get_ext_type(x):
    return x.split(".")[1] if "." in x else None


def _get_ext_det(x):
    return x.split(".")[0] if "." in x else x

In [ ]:
# | export


def create_zarr_ref_from_fits(fns, return_wcs=False, outfn=None):
    """Create a zarr reference from a set of Euclid fits files.

    This uses kerchunk to build a zarr reference file for accessing data directly from fits files.
    The resulting reference can then be opened as a xarray Dataset with `open_zarr_ref_as_dataset`.
    """
    if isinstance(fns, str):
        fns = [fns]
    else:
        fns = [str(fn) for fn in fns]
    fn = fns[0]
    if "MER" in str(fn):
        product_id_name = "tile_index"
        get_product_id = get_tile_index_from_filename
    else:
        product_id_name = "observation_id"
        get_product_id = get_obs_id_from_filename
    with fits.open(fn) as hdul:
        exts = list(h.name for h in hdul if h.size > 0)
    ref_files = []
    product_ids = []
    dither_ids = []
    filters = []
    wcs_list = []
    pbar = tqdm(fns)
    for fn in pbar:
        product_id = get_product_id(fn)
        product_ids.append(product_id)
        dither_id = get_dither_id_from_filename(fn)
        if dither_id is not None:
            dither_ids.append(dither_id)
        filter = get_filter_from_filename(fn)
        if dither_id is not None:
            filters.append(filter)
        pbar.set_description(
            f"{fn.split("/")[-1]}, {product_id}, {dither_id}, {filter}"
        )
        ref_exts = []
        coords = []
        coord_name = "extension"
        for ext in exts:
            ext_type = _get_ext_type(ext)
            out = process_file(fn, extension=ext)
            if ext_type is not None:
                out = _rename(out, ext_type)
                coord_name = "detector"
            out = _fix_byte_order(out)
            ref_exts.append(out)
            coord = _get_ext_det(ext)
            coords.append(coord)
            hdr = fits.getheader(fn, ext)
            if return_wcs:
                wcs = {
                    k: hdr[k]
                    for k in (
                        "CTYPE1",
                        "CTYPE2",
                        "CD1_1",
                        "CD1_2",
                        "CD2_1",
                        "CD2_2",
                        "CRPIX1",
                        "CRPIX2",
                        "CRVAL1",
                        "CRVAL2",
                        "NAXIS1",
                        "NAXIS2",
                    )
                }
                wcs[coord_name] = coord
                if dither_id is not None:
                    wcs["dither"] = dither_id
                if dither_id is not None:
                    wcs[product_id_name] = product_id
                wcs_list.append(wcs)
        mzz = MultiZarrToZarr(
            ref_exts,
            coo_map={coord_name: coords},
            concat_dims=[coord_name],
            identical_dims=["x", "y"],
        )
        ref_exts = mzz.translate()
        ref_files.append(ref_exts)
    coo_map = {}
    concat_dims = []
    if len(np.unique(product_ids)) > 0:
        coo_map[product_id_name] = product_ids
        concat_dims.append(product_id_name)
    if len(np.unique(dither_ids)) > 0:
        coo_map["dither"] = dither_ids
        concat_dims.append("dither")
    if len(np.unique(filters)) > 0:
        coo_map["filter"] = filters
        concat_dims.append("filter")
    mzz = MultiZarrToZarr(
        ref_files, coo_map=coo_map, concat_dims=concat_dims, identical_dims=["x", "y"]
    )
    with catch_warnings():
        filterwarnings(
            "ignore", "Concatenated coordinate .* contains less than expected"
        )
        out = mzz.translate()
    if outfn is not None:
        with open(outfn, "w") as f:
            json.dump(out, f)
    if return_wcs:
        wcs = pd.DataFrame(wcs_list)
        wcs = wcs.groupby([coord_name, "dither", product_id_name]).first()
        wcs = wcs.to_xarray()
        return out, wcs
    else:
        return out

In [ ]:
# | export


def open_zarr_ref_as_dataset(ref):
    """Open a zarr reference as a dask xarray Dataset."""
    ds = xr.open_dataset(
        "reference://",
        engine="zarr",
        chunks="auto",
        backend_kwargs={
            "storage_options": {
                "fo": ref,
            },
            "consolidated": False,
        },
    )
    return ds

In [ ]:
# | export


def open_fits_as_dataset(fns):
    """Create a zarr reference from a set of fits files and open as a dask xarray Dataset.

    If you plan to reopen the Dataset repeatedly, then is would be better to create the
    JSON reference once with `create_zarr_ref_from_fits`, save it, and then just open
    the reference as needed with `open_zarr_ref_as_dataset`.
    """
    ref = create_zarr_ref_from_fits(fns)
    ds = open_zarr_ref_as_dataset(ref)
    return ds

In [ ]:
# | export


def write_da_to_fits(
    da,  # the DataArray, with coordinates `x`, `y`, and `detector`
    fn,  # the filename to which to write the data
    da_wcs=None,  # an optional DataArray of WCS info for each `detector`
    overwrite=False,  # should any existing file be overwritten
):
    """Write a DataArray to a FITS file."""
    hdul = HDUList(PrimaryHDU())
    if "detector" in da.coords:
        for det in da["detector"].to_numpy():
            if da_wcs is not None:
                wcs = WCS(da_wcs.sel(detector=det).to_pandas())
                hdr = wcs.to_header()
            else:
                hdr = None
            hdul.append(ImageHDU(da.sel(detector=det), name=det, header=hdr))
    else:
        if da_wcs is not None:
            wcs = WCS(da_wcs.to_pandas())
            hdr = wcs.to_header()
        else:
            hdr = None
        hdul.append(ImageHDU(da, header=hdr))
    hdul.writeto(fn, overwrite=overwrite)
    return hdul

In [ ]:
# | export


def load_zarr_ref_from_file(zarrfn):
    with open(zarrfn) as f:
        ref = json.load(f)
    return ref

## Example

In [ ]:
path = default_data_path("Q1_R1")
image_info = get_nisp_images_for_observation(2683, path=path)
fns = image_info.filename

In [ ]:
%%time
ref, wcs = create_zarr_ref_from_fits(fns, return_wcs=True)

/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_J-2683-0_20240930T172941.852898Z.fits 2683 0 J
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_H-2683-0_20240930T184607.344746Z.fits 2683 0 H
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_Y-2683-0_20240930T174927.996060Z.fits 2683 0 Y
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_J-2683-1_20240930T172941.702020Z.fits 2683 1 J
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_H-2683-1_20240930T184607.455806Z.fits 2683 1 H
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_Y-2683-1_20240930T174928.133198Z.fits 2683 1 Y
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_J-2683-2_20240930T172941.856270Z.fits 2683 2 J
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_H-2683-2_20240930T184607.453358Z.fits 2683 2 H
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMAGE_Y-2683-2_20240930T174928.131907Z.fits 2683 2 Y
/Users/spb/euclid_data/Q1_R1/NIR/2683/EUC_NIR_W-CAL-IMA

In [ ]:
wcs

<xarray.Dataset> Size: 6kB
Dimensions:         (detector: 16, dither: 4, observation_id: 1)
Coordinates:
  * detector        (detector) object 128B 'DET11' 'DET12' ... 'DET43' 'DET44'
  * dither          (dither) int64 32B 0 1 2 3
  * observation_id  (observation_id) int64 8B 2683
Data variables:
    CTYPE1          (detector, dither, observation_id) object 512B 'RA---TPV'...
    CTYPE2          (detector, dither, observation_id) object 512B 'DEC--TPV'...
    CD1_1           (detector, dither, observation_id) float64 512B -4.317e-0...
    CD1_2           (detector, dither, observation_id) float64 512B -7.021e-0...
    CD2_1           (detector, dither, observation_id) float64 512B 7.021e-05...
    CD2_2           (detector, dither, observation_id) float64 512B -4.317e-0...
    CRPIX1          (detector, dither, observation_id) float64 512B 4.17e+03 ...
    CRPIX2          (detector, dither, observation_id) float64 512B 4.621e+03...
    CRVAL1          (detector, dither, observation_id) float64 512B 268.5 ......
    CRVAL2          (detector, dither, observation_id) float64 512B 68.23 ......
    NAXIS1          (detector, dither, observation_id) int64 512B 2040 ... 2040
    NAXIS2          (detector, dither, observation_id) int64 512B 2040 ... 2040

In [ ]:
WCS(wcs.sel(detector="DET11", dither=2, observation_id=2683).to_pandas())

WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN' 
CRVAL : 268.6563503943 68.24617668136 
CRPIX : 4170.390230645 4621.116938306 
CD1_1 CD1_2  : -4.297985528424e-05 -7.032797254342e-05 
CD2_1 CD2_2  : 7.03307850190499e-05 -4.29881884951e-05 
NAXIS : 2040  2040

In [ ]:
%%time
ds = open_zarr_ref_as_dataset(ref)

CPU times: user 297 ms, sys: 130 ms, total: 427 ms
Wall time: 1.15 s


In [ ]:
ds

<xarray.Dataset> Size: 10GB
Dimensions:         (observation_id: 1, dither: 4, filter: 3, detector: 16,
                     y: 2040, x: 2040)
Coordinates:
  * detector        (detector) object 128B 'DET11' 'DET12' ... 'DET43' 'DET44'
  * dither          (dither) int64 32B 0 1 2 3
  * filter          (filter) object 24B 'H' 'J' 'Y'
  * observation_id  (observation_id) int64 8B 2683
Dimensions without coordinates: y, x
Data variables:
    DQ              (observation_id, dither, filter, detector, y, x) int32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>
    RMS             (observation_id, dither, filter, detector, y, x) float32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>
    SCI             (observation_id, dither, filter, detector, y, x) float32 3GB dask.array<chunksize=(1, 2, 2, 2, 2040, 2040), meta=np.ndarray>

In [ ]:
median = ds["SCI"].median(dim=["observation_id", "dither", "filter"])

In [ ]:
%%time
median.compute()

CPU times: user 1min 34s, sys: 1min 54s, total: 3min 29s
Wall time: 37.6 s


<xarray.DataArray 'SCI' (detector: 16, y: 2040, x: 2040)> Size: 266MB
array([[[ 64.19491  ,  67.078766 ,  66.44539  , ...,  64.04256  ,
          62.787518 ,  59.350792 ],
        [ 64.54579  ,  63.714226 ,  60.695328 , ...,  62.01454  ,
          63.63827  ,  60.694565 ],
        [ 60.816174 ,  61.2768   ,  63.714226 , ...,  66.12628  ,
          55.953312 ,  55.2466   ],
        ...,
        [ 63.453663 ,  67.74246  ,  62.702446 , ...,  59.646862 ,
          62.916855 ,  63.85218  ],
        [ 64.9606   ,  59.288826 ,  63.397884 , ...,  63.160946 ,
          63.157394 ,  65.6652   ],
        [ 59.76058  ,  60.67249  ,  68.97561  , ...,  62.60383  ,
          58.301983 ,  62.16313  ]],

       [[ 70.7383   ,  69.91258  ,  72.29007  , ...,  -8.780915 ,
          -5.9143744,  -6.920571 ],
        [ 69.49198  ,  67.1319   ,  75.94953  , ...,  -4.9328685,
         -12.070469 , -10.73223  ],
        [ 71.27069  ,  71.91942  ,  63.27051  , ...,  -4.968932 ,
          -4.938717 ,  -5.9631076],
...
        [ 72.38832  ,  67.58812  ,  69.579926 , ..., -19.608631 ,
         -10.109669 ,  -8.403699 ],
        [ 66.27293  ,  69.674515 ,  61.857292 , ...,  -9.210991 ,
         -16.680946 , -10.271188 ],
        [ 69.171005 ,  69.53067  ,  62.534737 , ..., -17.500885 ,
         -15.873654 ,  -7.4193735]],

       [[ 76.75749  ,  83.60602  ,  83.27513  , ...,  67.20813  ,
          62.931004 ,  68.58031  ],
        [ 74.80112  ,  72.93109  ,  80.6994   , ...,  64.274536 ,
          61.619118 ,  65.80408  ],
        [ 76.75749  ,  74.95543  ,  78.44336  , ...,  71.57124  ,
          72.47168  ,  70.66237  ],
        ...,
        [ 68.75368  ,  69.03995  ,  65.620316 , ...,  68.860085 ,
          64.969086 ,  63.070034 ],
        [ 66.44855  ,  64.932816 ,  72.47544  , ...,  64.05542  ,
          59.470894 ,  71.957886 ],
        [ 68.491776 ,  64.79114  ,  68.05373  , ...,  68.38541  ,
          65.33897  ,  66.87998  ]]], dtype=float32)
Coordinates:
  * detector  (detector) object 128B 'DET11' 'DET12' 'DET13' ... 'DET43' 'DET44'
Dimensions without coordinates: y, x

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()